# Example: Relinearising

As explained in Example 5, it is possible to simulative evolutive scenarios using the linear solver (`LD+GS`). Here, the dynamics of the plasma are linearised by creating jacobian matrices showing how the plasma responds to changes in the coil/metal currents and the plasma profile parameters. Over time, this linearisation can degrade and no longer accurately approximates the current plasma behaviour. Relinearisation is the act of re-calculating the jacobian matrices at a set interval or according to some other conditions. This example notebook will use the scenario from example 5 but will show how relinearisation greatly improves the agreement between non-linear and linear simulations.


## Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import pickle

## Build the machine

In [ ]:
# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

### Instantiate an equilibrium

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,   # Radial range
    Zmin=-2.2, Zmax=2.2,  # Vertical range
    nx=65,                # Number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # Number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)  

### Instatiate a profile object

In [ ]:
from freegsnke.jtor_update import ConstrainPaxisIp

profiles = ConstrainPaxisIp(
    eq=eq,        # equilibrium object
    paxis=8.1e3,  # profile object
    Ip=6.2e5,     # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

### Set coil currents
Here we set coil currents that create a diverted plasma (as seen in example 5). 

In [ ]:
with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
    current_values = pickle.load(f)

for key in current_values.keys():
    eq.tokamak.set_coil_current(coil_label=key, current_value=current_values[key])

### Instatiate the solver


In [ ]:
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)    

### Call forward solver to find equilibrium 

In [ ]:
GSStaticSolver.solve(
    eq=eq, 
    profiles=profiles, 
    constrain=None, 
    target_relative_tolerance=1e-8,
    verbose=0
)

### Plot the initial equilibrium 

In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

### Define descriptors

Descriptors are quantaties of interest that will be approximated via a linearisation from the initial equilibrium. This means it will be possible to find key time-dependant quantities of interest e.g. `zIp` without solving the computationally expensive Grad-Shafranov equation.

In [ ]:
# choose the x-point in XPT_BOX for calculating
# certain plasma descriptors 
XPT_BOX = [[0.33, -0.88], [0.95, -1.38]]

def plasma_descriptors(eq):
    xpt_mask = (
        (eq.xpt[:, 0] >= XPT_BOX[0][0])
        & (eq.xpt[:, 0] <= XPT_BOX[1][0])
        & (eq.xpt[:, 1] <= XPT_BOX[0][1])
        & (eq.xpt[:, 1] >= XPT_BOX[1][1])
    )

    xpts = eq.xpt[xpt_mask, 0:2].squeeze()
    if xpts.ndim > 1 and xpts.shape[0] > 1:
        opt = eq.opt[0, 0:2]
        dists = np.linalg.norm(xpts - opt, axis=1)
        idx = np.argmin(dists)  # index of closest point
        rx, zx = xpts[idx, :]
    else:
        rx, zx = xpts

    Rin, Rout = eq.innerOuterSeparatrix()

    # zIp, Rx, Zx, Rin, Rout
    return np.array(
        [eq.Zcurrent(), rx, zx, Rin, Rout]
    )

### Time evolution

We are now ready to solve the forward time-evolutive problem. This will follow the same setup as example 05.

In [ ]:
from freegsnke import nonlinear_solve

def create_nl_solver():

    return nonlinear_solve.nl_solver(
        eq=eq, 
        profiles=profiles, 
        GSStaticSolver=GSStaticSolver,
        full_timestep=5e-4, 
        plasma_resistivity=1e-6,
        max_mode_frequency=10**2.5,
        plasma_descriptor_function=plasma_descriptors
    )

stepping = create_nl_solver()

#### Set active coil voltages

Again, like example 05, we will evolve the plasma with no control or external drive.

In [ ]:
voltages = (stepping.vessel_currents_vec*stepping.evol_metal_curr.coil_resist)[:stepping.evol_metal_curr.n_active_coils] 

To start, the solver is prepared by setting the initial conditions.

In [ ]:
stepping.initialize_from_ICs(eq, profiles)

In [ ]:
initial_maps = []
new_maps = []
for i in range(stepping.n_active_coils + 10):
    initial_maps.append(
        profiles.limiter_handler.rebuild_map2d(stepping.dIydI[:,i], eq.R, profiles.limiter_handler.idxs_mask)
    )

#### Set time steps
Now we set the total number of time steps we want to simulate

In [ ]:
# number of time steps to simulate
max_count = 50

# initialising some variables for iteration and logging
counter = 0
t = 0

#### Set time-dependent active coil voltages, profile parameters, and plasma resistivity


In [ ]:
voltages = (stepping.vessel_currents_vec*stepping.evol_metal_curr.coil_resist)[:stepping.evol_metal_curr.n_active_coils] 

# we define some time-dependent plasma current density profile parameters here
alpha_m = np.tile(profiles.alpha_m, max_count+1)
alpha_m -= (0.1 * np.sin(0.05 * np.pi * np.arange(max_count+1))) # we add some perturbation

alpha_n = np.tile(profiles.alpha_n, max_count+1)
alpha_n += (0.1 * np.sin(0.1 * np.pi * np.arange(max_count+1))) # we add some perturbation

paxis = np.tile(profiles.paxis, max_count+1)
paxis += (0.1 * np.sin(0.01 * np.pi * np.arange(max_count+1))) # we add some perturbation

#### Call the solver (linear)
Finally, we call the time-evolutive solver with `stepping.nlstepper()` sequentially until we reach the preset end time.

Every 5 timesteps we set `relinearise` to `True`, causing the jacobian matrices to be re-calculated.

In [ ]:
# initialise the solver with the initial equilibrium/profiles
stepping.initialize_from_ICs(eq, profiles)

history_plasma_descriptors_relinearised = [plasma_descriptors(stepping.eq1)]
history_times_relinearised = [t]
history_currents_relinearised = [stepping.currents_vec]
history_equilibria_relinearised = [stepping.eq1.create_auxiliary_equilibrium()]

In [ ]:
relinearisation_stored = False
# loop over time steps
while counter<max_count:
    print(f'Step: {counter}/{max_count-1}')
    print(f'--- t = {t:.2e}')


    old_dIydI = stepping.dIydI.copy()

    # carry out the time step (profile parameters and resistivity are constant at each time step)
    stepping.nlstepper(
        active_voltage_vec=voltages,   # same voltages used at each time step
        linear_only=True,
        no_GS=False,
        verbose=False,
        max_solving_iterations=50,
        profiles_parameters = {
            "alpha_m": alpha_m[counter],
            "alpha_n": alpha_n[counter],
            "paxis": paxis[counter],
        },
        relinearise_threshold=0.05
    )

    if (old_dIydI != stepping.dIydI).any():
        for i in range(stepping.n_active_coils + 10):
            new_maps.append(
                profiles.limiter_handler.rebuild_map2d(stepping.dIydI[:,i], eq.R, profiles.limiter_handler.idxs_mask)
            )
        relinearisation_stored = True

    # store information on the time step
    t += stepping.dt_step
    history_times_relinearised.append(t)
    counter += 1

    # store time-advanced equilibrium, currents, and profiles (+ other quantites of interest)
    history_currents_relinearised.append(stepping.currents_vec)
    history_equilibria_relinearised.append(stepping.eq1.create_auxiliary_equilibrium())
    history_plasma_descriptors_relinearised.append(plasma_descriptors(stepping.eq1))

# transform lists to arrays
history_currents_relinearised = np.array(history_currents_relinearised)
history_times_relinearised = np.array(history_times_relinearised)
history_plasma_descriptors_relinearised = np.array(history_plasma_descriptors_relinearised)


We can visualise how the jacobian matrix $\frac{\partial I_y}{\partial I}$ changes by looking at the 2D core map before and after (and the difference between the two).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(initial_maps), 3, figsize=(25, 150))

for idx in range(len(initial_maps)):
    maximum = max(initial_maps[idx].max(), new_maps[idx].max())
    minimum = max(initial_maps[idx].min(), new_maps[idx].min())

    sns.heatmap(initial_maps[idx], ax=axes[idx, 0], vmax=maximum, vmin=minimum)
    sns.heatmap(initial_maps[idx] - new_maps[idx], ax=axes[idx, 1], vmax=maximum, vmin=minimum)
    sns.heatmap(new_maps[idx], ax=axes[idx, 2], vmax=maximum, vmin=minimum)


    axes[idx, 0].set_title(f"dIydI_0 (mode={idx+1})")
    axes[idx, 1].set_title(f"dIydI_0 - dIydI_1 (mode={idx+1})")
    axes[idx, 2].set_title(f"dIydI_1 (mode={idx+1})")

#### Call the solver (non-linear)

We use the fully non-linear solver to calculate the exact evolution of the plasma.

In [ ]:
# reset the solver object
stepping = create_nl_solver()
stepping.initialize_from_ICs(eq, profiles)

# initialising some variables for iteration and logging
counter = 0
t = 0

# store the true descriptors this time, not an approximation
history_plasma_descriptors_linear = [plasma_descriptors(stepping.eq1)]

history_times_linear = [t]
history_currents_linear = [stepping.currents_vec]
history_equilibria_linear = [stepping.eq1.create_auxiliary_equilibrium()]

# loop over the time steps
while counter<max_count:

    print(f'Step: {counter}/{max_count-1}')
    print(f'--- t = {t:.2e}')
    
    # carry out the time step
    stepping.nlstepper(
        active_voltage_vec=voltages, 
        linear_only=False,
        no_GS=False,
        verbose=False,
        max_solving_iterations=50,
        profiles_parameters = {
            "alpha_m": alpha_m[counter],
            "alpha_n": alpha_n[counter],
            "paxis": paxis[counter],
        },
    )

    # store information on the time step
    t += stepping.dt_step
    history_times_linear.append(t)
    counter += 1
    
    # store time-advanced equilibrium, currents, and profiles (+ other quantites of interest)
    history_currents_linear.append(stepping.currents_vec)
    history_equilibria_linear.append(stepping.eq1.create_auxiliary_equilibrium())
    history_plasma_descriptors_linear.append(plasma_descriptors(stepping.eq1))
    

# transform lists to arrays
history_currents_linear = np.array(history_currents_linear)
history_times_linear = np.array(history_times_linear)
history_plasma_descriptors_linear = np.array(history_plasma_descriptors_linear)



### Plotting
Plot a comparison of the linear solve (with re-linearisation) and the non-linear solver for several plasma quantities. It shows that the linear simulation and the non-linear simulation agree remarkably well. 

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(15, 5))

ax[0].plot(history_times_relinearised, history_plasma_descriptors_relinearised[:, 0], label="Linear (Relinearised)")
ax[0].plot(history_times_relinearised, history_plasma_descriptors_linear[:, 0], label="Non-linear")
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("ZCurrent")
ax[0].legend()

ax[1].plot(history_times_relinearised, history_plasma_descriptors_relinearised[:, 1], label="Linear (Relinearised)")
ax[1].plot(history_times_relinearised, history_plasma_descriptors_linear[:, 1], label="Non-linear")
ax[1].set_xlabel("Time (s)")
ax[1].set_ylabel("Rx (m)")
ax[1].legend()

ax[2].plot(history_times_relinearised, history_plasma_descriptors_relinearised[:, 2], label="Linear (Relinearised)")
ax[2].plot(history_times_relinearised, history_plasma_descriptors_linear[:, 2], label="Non-linear")
ax[2].set_xlabel("Time (s)")
ax[2].set_ylabel("Zx (m)")
ax[2].legend()

ax[3].plot(history_times_relinearised, history_plasma_descriptors_relinearised[:, 3], label="Linear (Relinearised)")
ax[3].plot(history_times_relinearised, history_plasma_descriptors_linear[:, 3], label="Non-linear")
ax[3].set_xlabel("Time (s)")
ax[3].set_ylabel("Rin (m)")
ax[3].legend()

ax[4].plot(history_times_relinearised, history_plasma_descriptors_relinearised[:, 4], label="Linear (Relinearised)")
ax[4].plot(history_times_relinearised, history_plasma_descriptors_linear[:, 4], label="Non-linear")
ax[4].set_xlabel("Time (s)")
ax[4].set_ylabel("Rout (m)")
ax[4].legend()

fig.tight_layout()